# Qdrant In-Memory Vector Store

This notebook demonstrates FFAI's Qdrant vector store backend in **in-memory mode**.

- No server, no files, no API keys required
- Data lives only in the process — gone when the kernel stops
- Ideal for prototyping, testing, and notebooks

We use synthetic embeddings (random unit vectors) so no external embedding
service is needed. The same workflow works with real embeddings from
`Embeddings(model='mistral/mistral-embed')`.

<div class="page-break"></div>

---

## Section 1: Setup

In [ ]:
import sys
from pathlib import Path

_cwd = Path().resolve()
_project_root = _cwd
for _p in [_cwd, *list(_cwd.parents)]:
    if (_p / 'pyproject.toml').is_file():
        _project_root = _p
        break

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

import numpy as np  # noqa: E402

from ffai.rag.stores import (  # noqa: E402
    get_store,
    is_store_available,
    list_available_stores,
)

print(f'Project root: {_project_root}')
print(f'Qdrant available: {is_store_available("qdrant")}')
print(f'Available backends: {list_available_stores()}')

<div class="page-break"></div>

---

## Section 2: Create the Store

`location=":memory:"` starts Qdrant entirely in RAM. No files are created.

In [ ]:
DIM = 128

store = get_store(
    backend='qdrant',
    collection_name='demo_memory',
    embedding_dim=DIM,
    location=':memory:',
)

print(f'Backend: {store.name}')
print('Collection: demo_memory')
print(f'Initial count: {store.count()}')

<div class="page-break"></div>

---

## Section 3: Add Documents

Generate synthetic embeddings (random unit vectors) and add them with metadata.

In [ ]:
def make_embedding(text: str, dim: int = DIM) -> list[float]:
    rng = np.random.default_rng(hash(text) & 0xFFFFFFFF)
    vec = rng.standard_normal(dim)
    return (vec / np.linalg.norm(vec)).tolist()


docs = [
    {'id': 'doc1_chunk0', 'text': 'Python asyncio provides asynchronous programming.',
     'source': 'python_async.md', 'chunking_strategy': 'recursive'},
    {'id': 'doc1_chunk1', 'text': 'Coroutines use async/await syntax for concurrent I/O.',
     'source': 'python_async.md', 'chunking_strategy': 'recursive'},
    {'id': 'doc2_chunk0', 'text': 'Rust ownership ensures memory safety without garbage collection.',
     'source': 'rust_ownership.md', 'chunking_strategy': 'recursive'},
    {'id': 'doc2_chunk1', 'text': 'The borrow checker enforces strict lifetime rules.',
     'source': 'rust_ownership.md', 'chunking_strategy': 'recursive'},
    {'id': 'doc3_chunk0', 'text': 'Go goroutines are lightweight concurrency primitives.',
     'source': 'go_goroutines.md', 'chunking_strategy': 'markdown'},
]

ids = [d['id'] for d in docs]
texts = [d['text'] for d in docs]
embeddings = [make_embedding(t) for t in texts]
metadatas = [{'source': d['source'], 'chunking_strategy': d['chunking_strategy']} for d in docs]

from ffai.rag._async import run_sync  # noqa: E402

added = run_sync(store.aadd(ids, texts, embeddings, metadatas))
print(f'Added: {added} chunks')
print(f'Total count: {store.count()}')
print(f'Sources: {store.list_sources()}')

<div class="page-break"></div>

---

## Section 4: Search

Search using a synthetic query embedding. The same embedding function
ensures queries about the same topic produce vectors near the stored ones.

In [ ]:
query_text = 'How does async programming work in Python?'
query_emb = make_embedding(query_text)

hits = run_sync(store.asearch(query_emb, top_k=3))

print(f'Query: {query_text}')
print(f'Top {len(hits)} results:\n')
for i, h in enumerate(hits, 1):
    print(f'  {i}. [{h.score:.4f}] {h.content}')
    print(f'     source={h.source}')
    print()

<div class="page-break"></div>

---

## Section 5: Filtered Search

Filter by metadata using the `where` parameter. Only results matching
`source=python_async.md` are returned.

In [ ]:
hits_filtered = run_sync(store.asearch(
    query_emb, top_k=3, where={'source': 'python_async.md'},
))

print(f'Filtered search (source=python_async.md): {len(hits_filtered)} results\n')
for i, h in enumerate(hits_filtered, 1):
    print(f'  {i}. [{h.score:.4f}] {h.content}')
    print(f'     source={h.source}')
    print()

<div class="page-break"></div>

---

## Section 6: Delete and Verify

Delete by source and verify the count drops.

In [ ]:
count_before = store.count()
store.delete_by_source('python_async.md')
count_after = store.count()

print(f'Before delete: {count_before} chunks')
print('Deleted source: python_async.md')
print(f'After delete: {count_after} chunks')
print(f'Remaining sources: {store.list_sources()}')

<div class="page-break"></div>

---

## Section 7: Cleanup

In-memory mode leaves no files on disk. Calling `clear()` resets the
collection. The data disappears when the kernel stops.

In [ ]:
store.clear()
print(f'After clear: {store.count()} chunks')
print('In-memory store leaves no files on disk.')
print('Data vanishes when the kernel process exits.')